# Artificial Intelligence Laboratory — Logical Planning

## Submission Notebook

This notebook contains the planning-domain specification, BFS planner, execution, and validation tests.


## Task 0 — Planning problem

### Initial state
`{At(Robot,A), At(Package,A)}`

### Goal
`{At(Package,C)}`

### Available actions
- `Move(A,B)`
- `Move(B,A)`
- `Move(B,C)`
- `Move(C,B)`
- `PickUp(Package,A)`
- `Drop(Package,C)`

An action is applicable only when all of its positive preconditions are satisfied and none of its negative preconditions is present.

The handout's illustrative sequence uses `PickUp(Package,B)` after `Move(A,B)`, but under the stated initial state and pickup preconditions that action is not applicable. The executable planner therefore uses the logically valid pickup action at A.

In [2]:
from collections import deque

class Action:
    def __init__(self, name, pos_pre=(), neg_pre=(), pos_eff=(), neg_eff=()):
        self.name = name
        self.pos_pre = frozenset(pos_pre)
        self.neg_pre = frozenset(neg_pre)
        self.pos_eff = frozenset(pos_eff)
        self.neg_eff = frozenset(neg_eff)

    def applicable(self, state):
        return self.pos_pre <= state and self.neg_pre.isdisjoint(state)

    def apply(self, state):
        if not self.applicable(state):
            raise ValueError(f"Action not applicable: {self.name}")
        return frozenset((state - self.neg_eff) | self.pos_eff)

    def __repr__(self):
        return self.name


initial = frozenset({
    "At(Robot,A)",
    "At(Package,A)"
})

goal = frozenset({
    "At(Package,C)"
})

actions = [
    Action("Move(A,B)",
           pos_pre=["At(Robot,A)"],
           pos_eff=["At(Robot,B)"],
           neg_eff=["At(Robot,A)"]),

    Action("Move(B,A)",
           pos_pre=["At(Robot,B)"],
           pos_eff=["At(Robot,A)"],
           neg_eff=["At(Robot,B)"]),

    Action("Move(B,C)",
           pos_pre=["At(Robot,B)"],
           pos_eff=["At(Robot,C)"],
           neg_eff=["At(Robot,B)"]),

    Action("Move(C,B)",
           pos_pre=["At(Robot,C)"],
           pos_eff=["At(Robot,B)"],
           neg_eff=["At(Robot,C)"]),

    Action("PickUp(Package,A)",
           pos_pre=["At(Robot,A)", "At(Package,A)"],
           pos_eff=["Holding(Package)"],
           neg_eff=["At(Package,A)"]),

    Action("Drop(Package,C)",
           pos_pre=["At(Robot,C)", "Holding(Package)"],
           pos_eff=["At(Package,C)"],
           neg_eff=["Holding(Package)"]),
]

In [3]:
def state_text(state):
    order = [
        "At(Robot,A)", "At(Robot,B)", "At(Robot,C)",
        "At(Package,A)", "At(Package,C)", "Holding(Package)"
    ]
    return ", ".join(x for x in order if x in state)


# Initial applicability checks
pickup = next(a for a in actions if a.name == "PickUp(Package,A)")
drop = next(a for a in actions if a.name == "Drop(Package,C)")

print("PickUp(Package,A) applicable:", pickup.applicable(initial))
print("Drop(Package,C) applicable:", drop.applicable(initial))

PickUp(Package,A) applicable: True
Drop(Package,C) applicable: False


In [4]:
def bfs_plan(start, goal_facts, actions):
    queue = deque([start])
    parent = {start: None}
    used_action = {}

    while queue:
        state = queue.popleft()

        if goal_facts <= state:
            plan = []
            states = []
            current = state

            while current is not None:
                states.append(current)
                if parent[current] is not None:
                    plan.append(used_action[current])
                current = parent[current]

            return list(reversed(plan)), list(reversed(states))

        for action in actions:
            if action.applicable(state):
                successor = action.apply(state)

                if successor not in parent:
                    parent[successor] = state
                    used_action[successor] = action
                    queue.append(successor)

    return None, None


def verify_plan(start, goal_facts, plan):
    state = start
    states = [state]

    for action in plan:
        if not action.applicable(state):
            return False, states
        state = action.apply(state)
        states.append(state)

    return goal_facts <= state, states

In [5]:
# Task 1 — Generate and display the plan

plan, states = bfs_plan(initial, goal, actions)

print("Plan found:", plan is not None)
print()

for i, state in enumerate(states):
    if i == 0:
        print(f"S{i}: {state_text(state)}")
    else:
        print(f"S{i}: after {plan[i-1]} -> {state_text(state)}")

print()
print("Actions:")
for i, action in enumerate(plan, start=1):
    print(f"{i}. {action}")

Plan found: True

S0: At(Robot,A), At(Package,A)
S1: after PickUp(Package,A) -> At(Robot,A), Holding(Package)
S2: after Move(A,B) -> At(Robot,B), Holding(Package)
S3: after Move(B,C) -> At(Robot,C), Holding(Package)
S4: after Drop(Package,C) -> At(Robot,C), At(Package,C)

Actions:
1. PickUp(Package,A)
2. Move(A,B)
3. Move(B,C)
4. Drop(Package,C)


In [6]:
# Independent verification of the generated plan

valid, verified_states = verify_plan(initial, goal, plan)

assert valid
assert verified_states == states
assert states[-1] == frozenset({
    "At(Robot,C)",
    "At(Package,C)"
})

assert [a.name for a in plan] == [
    "PickUp(Package,A)",
    "Move(A,B)",
    "Move(B,C)",
    "Drop(Package,C)"
]

print("Independent plan verification passed.")

Independent plan verification passed.


In [7]:
# Task 3 — Test A: Solvable problem

test_a_plan, test_a_states = bfs_plan(initial, goal, actions)

assert test_a_plan is not None
assert verify_plan(initial, goal, test_a_plan)[0]

print("Test A: PASS")
print("Plan:", [a.name for a in test_a_plan])
print("Valid:", verify_plan(initial, goal, test_a_plan)[0])

Test A: PASS
Plan: ['PickUp(Package,A)', 'Move(A,B)', 'Move(B,C)', 'Drop(Package,C)']
Valid: True


In [8]:
# Task 3 — Test B: Impossible problem
# Remove the PickUp action.

actions_without_pickup = [
    a for a in actions
    if not a.name.startswith("PickUp")
]

test_b_plan, test_b_states = bfs_plan(
    initial, goal, actions_without_pickup
)

assert test_b_plan is None

print("Test B: PASS")
print("Planner output:", "No plan found")

Test B: PASS
Planner output: No plan found


In [9]:
# Task 3 — Test C: Irrelevant actions
# Add a movement action that changes the robot's location but not the package.

irrelevant_action = Action(
    "Move(A,B)-Irrelevant",
    pos_pre=["At(Robot,A)"],
    pos_eff=["At(Robot,B)"],
    neg_eff=["At(Robot,A)"]
)

actions_with_irrelevant = actions + [irrelevant_action]

test_c_plan, test_c_states = bfs_plan(
    initial, goal, actions_with_irrelevant
)

assert test_c_plan is not None
assert verify_plan(initial, goal, test_c_plan)[0]
assert "At(Package,C)" in test_c_states[-1]

print("Test C: PASS")
print("Plan:", [a.name for a in test_c_plan])
print("Final state:", state_text(test_c_states[-1]))
print("Robot reaching C alone was not treated as the goal.")

Test C: PASS
Plan: ['PickUp(Package,A)', 'Move(A,B)', 'Move(B,C)', 'Drop(Package,C)']
Final state: At(Robot,C), At(Package,C)
Robot reaching C alone was not treated as the goal.


In [10]:
# Map the implementation to the specification

print("Preconditions -> Action.applicable()")
print("Effects       -> Action.apply()")
print("Goal          -> goal_facts <= state")
print("BFS           -> queue + parent/used_action dictionaries")
print("Verification  -> verify_plan()")

Preconditions -> Action.applicable()
Effects       -> Action.apply()
Goal          -> goal_facts <= state
BFS           -> queue + parent/used_action dictionaries
Verification  -> verify_plan()
